# PubMed Retrieval with ModernPubMedBERT + FAISS

This notebook builds a FAISS index over `text_to_embed` using the Hugging Face model `lokeshch19/ModernPubMedBERT` and provides a simple retrieval function.

Notes:
- The model has a max sequence length of 2048 tokens. We enforce truncation to this limit.
- Embeddings use mean pooling over the last hidden state with attention mask.

In [8]:
import json
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss
import logging

In [9]:

LOGGER = logging.getLogger("retrieval_parser")

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

DATA_PATH = HERE /"preprocessing_output/semaglutide_pubmed.jsonl"
OUT_DIR = HERE / "retrieval_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOGGER_PATH = HERE / "logs/input_parser.log"

logging.basicConfig(
    filename=LOGGER_PATH,
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)


MODEL_NAME = "lokeshch19/ModernPubMedBERT"
MAX_LEN = 2048
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
from pymed import PubMed
import time
import re

# PubMed API helpers (legacy from earlier pipeline)
# Requires PUBMED_CFG in config.json with "tool" and "email"
PUBMED_CFG = json.load(open(HERE / "config.json", "r", encoding="utf-8"))["pubmed"]

pubmed = PubMed(
    tool=PUBMED_CFG["tool"],
    email=PUBMED_CFG["email"]
)

def normalize_year(year_value) -> int:
    """Coerce a variety of year representations to an int; return 0 if unknown."""
    if isinstance(year_value, int):
        return year_value
    if isinstance(year_value, str):
        try:
            return int(year_value)
        except Exception:
            m = re.search(r"\d{4}", year_value)
            return int(m.group(0)) if m else 0
    try:
        y = getattr(year_value, "year", None)
        return int(y) if isinstance(y, int) else 0
    except Exception:
        return 0

def fetch_articles(query: str, max_results: int):
    """Fetch PubMed articles by query string using pymed."""
    results = pubmed.query(query, max_results=max_results)
    articles = []
    sleep_s = PUBMED_CFG.get("sleep_between_requests", 0.34)

    for article in results:
        art = article.toDict()
        pub_date = art.get("publication_date")
        year = None
        if pub_date is not None:
            try:
                year = getattr(pub_date, "year", pub_date)
            except Exception:
                pass

        art["year"] = normalize_year(year)

        if art.get("publication_types") is None:
            art["publication_types"] = []

        if isinstance(art.get("abstract"), list):
            art["abstract"] = " ".join(art["abstract"])

        articles.append(art)
        time.sleep(sleep_s)

    return articles

def fetch_articles_by_ids(pubmed_ids):
    """Fetch articles directly by PubMed ID."""
    articles = []
    for pid in (pubmed_ids or []):
        q = f"{pid}[PMID]"
        fetched = fetch_articles(q, max_results=1)
        if fetched:
            articles.append(fetched[0])
    return articles


In [10]:
def load_records(path: Path):
    """Load JSONL records from disk.

Pydoc:
    Reads a JSONL file and returns a list of parsed JSON objects, skipping blank lines.

Args:
    path (Path): Path to the JSONL file.

Returns:
    list[dict]: Parsed records in file order.
"""
    records = []
    try:
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    LOGGER.error("JSON decode error in %s at line %d", path, line_no)
                    raise
    except FileNotFoundError:
        LOGGER.error("Input file not found: %s", path)
        raise
    return records

records = load_records(DATA_PATH)
LOGGER.info("Loaded %d records from %s", len(records), DATA_PATH)
len(records)

JSONDecodeError: Extra data: line 1 column 5316 (char 5315)

In [ ]:
def get_text(rec: dict) -> str:
    """Extract the text used for embedding from a record.

Pydoc:
    Pulls the `text_to_embed` field and returns a trimmed string.

Args:
    rec (dict): Record containing embedding text.

Returns:
    str: Cleaned text string (may be empty).
"""
    return (rec.get("text_to_embed") or "").strip()

texts = [get_text(r) for r in records]
keep = [i for i,t in enumerate(texts) if t]
removed = len(texts) - len(keep)
records = [records[i] for i in keep]
texts = [texts[i] for i in keep]
LOGGER.info("Filtered empty texts: kept=%d removed=%d", len(texts), removed)
len(texts)

1572

In [7]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
except OSError as exc:
    LOGGER.error("Failed to load model or tokenizer: %s", MODEL_NAME)
    raise
model.to(device)
model.eval()

def count_tokens(texts):
    """Count tokens for each text with the active tokenizer.

Pydoc:
    Uses the loaded Hugging Face tokenizer to compute token counts per input string.

Args:
    texts (list[str]): Input strings to tokenize.

Returns:
    list[int]: Token counts aligned with input order.
"""
    return [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts]

lengths = count_tokens(texts)
over = sum(1 for n in lengths if n > MAX_LEN)
max_len = max(lengths) if lengths else 0
LOGGER.info("Token lengths: max=%d over_max=%d (MAX_LEN=%d)", max_len, over, MAX_LEN)
over, max_len

NameError: name 'texts' is not defined

In [ ]:
@torch.no_grad()
def embed_texts(batch_texts):
    """Embed a batch of texts using the transformer model.

Pydoc:
    Tokenizes, runs the model, mean-pools over tokens, and returns CPU numpy embeddings.

Args:
    batch_texts (list[str]): Text batch to embed.

Returns:
    numpy.ndarray: Float32 embeddings with shape (batch, dim).
"""
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    last = out.last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1)
    summed = (last * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)
    mean = summed / counts
    return mean.cpu().numpy()

embs = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    embs.append(embed_texts(batch))

embs = np.vstack(embs).astype("float32")
LOGGER.info("Embeddings shape=%s dtype=%s", embs.shape, embs.dtype)
embs.shape

(1572, 768)

In [ ]:
dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)

# Normalize for cosine similarity with IP
faiss.normalize_L2(embs)
try:
    index.add(embs)
except RuntimeError:
    LOGGER.error("Failed to add embeddings to FAISS index")
    raise
LOGGER.info("FAISS index built: ntotal=%d", index.ntotal)

index.ntotal

1572

In [ ]:
# Save index + metadata
index_path = OUT_DIR / "semaglutide_pubmed_faiss.index"
meta_path = OUT_DIR / "semaglutide_pubmed_meta.jsonl"

faiss.write_index(index, str(index_path))

with meta_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=True) + "\n")

index_path, meta_path

(PosixPath('/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs/semaglutide_pubmed_faiss.index'),
 PosixPath('/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs/semaglutide_pubmed_meta.jsonl'))

In [ ]:
def retrieve(query: str, k: int = 5):
    """Retrieve top-k records for a query using FAISS.

Pydoc:
    Embeds the query, runs vector search, and returns a list of scored metadata dicts.

Args:
    query (str): Search query.
    k (int, optional): Number of results to return. Defaults to 5.

Returns:
    list[dict]: Results with score and selected record fields.
"""
    q_emb = embed_texts([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    try:
        scores, idxs = index.search(q_emb, k)
    except RuntimeError:
        LOGGER.error("FAISS search failed for k=%d", k)
        raise
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        rec = records[int(idx)]
        results.append({
            "score": float(score),
            "pmid": rec.get("pmid"),
            "doi": rec.get("doi"),
            "title": rec.get("title"),
            "journal": rec.get("journal"),
            "year": rec.get("year"),
            "abstract": rec.get("abstract"),
            "text_to_embed": rec.get("text_to_embed"),
        })
    return results

In [ ]:
#retrieve("Semaglutide induces significant weight loss in adults with obesity.", k=15)

In [ ]:
#HARDCODED_QUERY = "Semaglutide induces significant weight loss in adults with obesity."
HARDCODED_QUERY = "Semaglutide is associated with neuropsychiatric side effects including depression and suicidal ideation."

results_path = OUT_DIR / "semaglutide_retrieval.jsonl"
with results_path.open("w", encoding="utf-8") as f:
    hits = retrieve(HARDCODED_QUERY, k=15)
    f.write(json.dumps({
        "query": HARDCODED_QUERY,
        "articles": hits,
    }, ensure_ascii=False) + "\n")

results_path


NameError: name 'OUT_DIR' is not defined